In [1]:
# === RICOSTRUZIONE: fonde TUTTE le ricevute -> magazzino + vista consolidata ===
from pathlib import Path
import pandas as pd, sqlite3, shutil

BASE     = Path("ISTAT")
RICEVUTE = BASE / "ricevute"
VISTE    = BASE / "viste"
RICEVUTE.mkdir(parents=True, exist_ok=True)
VISTE.mkdir(exist_ok=True)

# ricevute sparse in root scivolano in ricevute/
for f in BASE.glob("fetch_full_*.csv"):
    dest = RICEVUTE / f.name
    if not dest.exists():
        shutil.move(str(f), str(dest))

files  = sorted(RICEVUTE.glob("fetch_full_*.csv"))
print("Ricevute fuse:", [f.name for f in files])
grezzo = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
grezzo = grezzo.drop_duplicates(subset=["REF_AREA", "SEX", "TIME_PERIOD"])

terr   = pd.read_csv(BASE / "territori_istat.csv")
nome   = dict(zip(terr["codice"].astype(str), terr["nome_it"].astype(str)))
parent = dict(zip(terr["codice"].astype(str), terr["parent"].astype(str)))
def livello(code):
    if code == "IT": return "Italia"
    p, c = 0, code
    while c != "IT" and p < 12:
        c = parent.get(c); p += 1
        if c is None or c == "nan": return "altro"
    return {1:"ripartizione", 2:"regione", 3:"provincia"}.get(p, "altro")

ez  = {1:"maschi", 2:"femmine", 9:"totale"}
cod = grezzo["REF_AREA"].astype(str)
lungo = pd.DataFrame({
    "codice":cod, "nome":cod.map(nome), "livello":cod.map(livello),
    "anno":grezzo["TIME_PERIOD"].astype(int),
    "sesso":grezzo["SEX"].astype(int).map(ez),
    "valore":grezzo["value"].astype(int),
})

con = sqlite3.connect(str(BASE / "istat.db"))
lungo.to_sql("popolazione", con, if_exists="replace", index=False)   # ora 'replace' riscrive TUTTO
rip = pd.read_sql_query("SELECT * FROM popolazione", con); con.close()

prov  = rip[rip.livello == "provincia"]
larga = prov.pivot(index=["codice","nome"], columns=["anno","sesso"], values="valore")
anni  = sorted({a for a,s in larga.columns})
larga = larga.reindex(columns=pd.MultiIndex.from_tuples(
    [(a,s) for a in anni for s in ["maschi","femmine","totale"]], names=["anno","sesso"]))
larga.to_csv(VISTE / "vista_province.csv")

sp = prov.pivot_table(index="anno", columns="sesso", values="valore", aggfunc="sum")
it = rip[rip.livello == "Italia"].pivot_table(index="anno", columns="sesso", values="valore")
print(f"Ricostruito: anni {anni}, vista {larga.shape} -> ISTAT/viste/vista_province.csv")
print("Province == Italia su ogni anno?", bool((sp.sort_index(axis=1) == it.sort_index(axis=1)).all().all()))


Ricevute fuse: ['fetch_full_2019.csv', 'fetch_full_2023.csv', 'fetch_full_2025.csv']
Ricostruito: anni [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], vista (107, 24) -> ISTAT/viste/vista_province.csv
Province == Italia su ogni anno? True
